In [0]:
from pyspark.sql import functions as F

base        = "/Volumes/workspace/finance/raw_files"
landing     = f"{base}/landing"        # where "new" statements arrive
checkpoints = f"{base}/_checkpoints"   # Auto Loader's memory

for m in ["04", "05", "06"]:
    dbutils.fs.cp(f"{base}/bank_a/bank_a_statement_2026_{m}.csv",
                  f"{landing}/bank_a/bank_a_statement_2026_{m}.csv")

display(dbutils.fs.ls(f"{landing}/bank_a"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/finance/raw_files/landing/bank_a/bank_a_statement_2026_04.csv,bank_a_statement_2026_04.csv,3100,1793773070000
dbfs:/Volumes/workspace/finance/raw_files/landing/bank_a/bank_a_statement_2026_05.csv,bank_a_statement_2026_05.csv,3282,1793773070000
dbfs:/Volumes/workspace/finance/raw_files/landing/bank_a/bank_a_statement_2026_06.csv,bank_a_statement_2026_06.csv,3188,1793773071000
dbfs:/Volumes/workspace/finance/raw_files/landing/bank_a/bank_a_statement_2026_07.csv,bank_a_statement_2026_07.csv,3348,1793772770000
dbfs:/Volumes/workspace/finance/raw_files/landing/bank_a/bank_a_statement_2026_08.csv,bank_a_statement_2026_08.csv,3081,1793772771000
dbfs:/Volumes/workspace/finance/raw_files/landing/bank_a/bank_a_statement_2026_09.csv,bank_a_statement_2026_09.csv,3180,1793772772000


In [0]:
def load_bronze_bank_a():
    table = "workspace.finance.bronze_bank_a"
    before = spark.table(table).count() if spark.catalog.tableExists(table) else 0

    stream = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .schema(schema_a)
        .load(f"{landing}/bank_a")
        .withColumn("source_file", F.col("_metadata.file_name"))
        .withColumn("ingested_at", F.current_timestamp()))

    (stream.writeStream
        .option("checkpointLocation", f"{checkpoints}/bronze_bank_a")
        .trigger(availableNow=True)
        .toTable(table)
        .awaitTermination())

    after = spark.table(table).count()
    print(f"bronze: {before} -> {after}  (+{after - before} rows)")

display(spark.sql("DESCRIBE HISTORY workspace.finance.bronze_bank_a")
        .select("version", "timestamp", "operation", "operationMetrics.numOutputRows"))

version,timestamp,operation,numOutputRows
3,2026-11-04T06:20:02.000Z,STREAMING UPDATE,3
2,2026-11-04T06:12:58.000Z,STREAMING UPDATE,103
1,2026-11-04T06:11:43.000Z,STREAMING UPDATE,102
0,2026-11-04T06:11:34.000Z,CREATE TABLE,null


In [0]:
display(spark.sql("""
  SELECT source_file, count(*) AS rows, max(ingested_at) AS ingested_at
  FROM workspace.finance.bronze_bank_a GROUP BY source_file ORDER BY source_file"""))

source_file,rows,ingested_at
bank_a_statement_2026_04.csv,33,2026-11-04T06:11:38.923Z
bank_a_statement_2026_05.csv,35,2026-11-04T06:11:38.923Z
bank_a_statement_2026_06.csv,34,2026-11-04T06:11:38.923Z
bank_a_statement_2026_07.csv,36,2026-11-04T06:12:55.155Z
bank_a_statement_2026_08.csv,33,2026-11-04T06:12:55.155Z
bank_a_statement_2026_09.csv,34,2026-11-04T06:12:55.155Z


In [0]:
load_bronze_bank_a()

rows written in this run: 0


In [0]:
for m in ["07", "08", "09"]:
    dbutils.fs.cp(f"{base}/bank_a/bank_a_statement_2026_{m}.csv",
                  f"{landing}/bank_a/bank_a_statement_2026_{m}.csv")

load_bronze_bank_a()


rows written in this run: 0


PART C

In [0]:
from pyspark.sql import Window

def to_amount(c):
    return F.expr(f"try_cast(regexp_replace(trim({c}), ',', '') AS DECIMAL(12,2))")

def clean_bank_a(df):                                   # Module 2 logic
    w, d = to_amount("withdrawal_raw"), to_amount("deposit_raw")
    return df.select(
        F.expr("to_date(try_to_timestamp(trim(date_raw), 'dd/MM/yy'))").alias("txn_date"),
        F.trim("narration").alias("description"),
        F.trim("ref_no").alias("bank_ref"),
        F.when(w > 0, w).otherwise(d).alias("amount"),
        F.when(w > 0, F.lit("DEBIT")).when(d > 0, F.lit("CREDIT")).alias("direction"),
        "source_file",
    ).withColumn("account", F.lit("bank_a"))

def add_txn_id(df):                                     # Module 4 logic
    key = F.concat_ws("|", "account", F.col("txn_date").cast("string"), F.upper("description"),
                      F.col("amount").cast("string"), "direction", F.coalesce("bank_ref", F.lit("")))
    w = Window.partitionBy("source_file", "dedupe_key").orderBy(F.lit(1))
    return (df.withColumn("dedupe_key", key)
              .withColumn("occurrence", F.row_number().over(w))
              .withColumn("txn_id", F.sha2(F.concat_ws("|", "dedupe_key", "occurrence"), 256))
              .drop("dedupe_key", "occurrence"))

In [0]:
def upsert_bank_a(batch_df, batch_id):
    valid = clean_bank_a(batch_df).filter(
        "txn_date IS NOT NULL AND amount > 0 AND direction IS NOT NULL")

    w = Window.partitionBy("txn_id").orderBy("source_file")
    new_rows = (add_txn_id(valid)
        .withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")   # dedupe within the batch
        .withColumn("loaded_at", F.current_timestamp()))

    new_rows.createOrReplaceTempView("new_rows")
    new_rows.sparkSession.sql("""
        MERGE INTO workspace.finance.silver_transactions AS t
        USING new_rows AS s
        ON t.txn_id = s.txn_id
        WHEN NOT MATCHED THEN INSERT *
    """)                                                                       # dedupe across batches

In [0]:
def load_silver_bank_a():
    (spark.readStream.table("workspace.finance.bronze_bank_a")     # only NEW bronze rows
        .writeStream
        .foreachBatch(upsert_bank_a)
        .option("checkpointLocation", f"{checkpoints}/silver_bank_a")
        .trigger(availableNow=True)
        .start()
        .awaitTermination())

before = spark.table("workspace.finance.silver_transactions").count()
load_silver_bank_a()
after = spark.table("workspace.finance.silver_transactions").count()
print(f"silver: {before} -> {after}")

silver: 272 -> 272


In [0]:
october = """Date,Narration,Chq./Ref.No.,Withdrawal Amt.,Deposit Amt.,Closing Balance
01/10/26,NEFT-N9988776655-ACME TECHNOLOGIES PVT LTD-SALARY OCT,900000000001,,"95,000.00","1,95,000.00"
05/10/26,IMPS-900000000002-RAJESH KUMAR-HOUSE RENT,900000000002,"18,000.00",,"1,77,000.00"
07/10/26,UPI/SWIGGY/swiggy@axisbank/Food order/900000003,900000000003,412.00,,"1,76,588.00"
"""
dbutils.fs.put(f"{landing}/bank_a/bank_a_statement_2026_10.csv", october, overwrite=True)

Wrote 359 bytes.


True

In [0]:
load_bronze_bank_a()
before = spark.table("workspace.finance.silver_transactions").count()
load_silver_bank_a()
after = spark.table("workspace.finance.silver_transactions").count()
print(f"silver: {before} -> {after}")    # expect +3

rows written in this run: 0
silver: 272 -> 275


In [0]:
display(spark.sql("DESCRIBE HISTORY workspace.finance.bronze_bank_a")
        .select("version", "timestamp", "operation", "operationMetrics"))

version,timestamp,operation,operationMetrics
3,2026-11-04T06:20:02.000Z,STREAMING UPDATE,"Map(numRemovedFiles -> 0, numOutputRows -> 3, numOutputBytes -> 3025, numAddedFiles -> 1)"
2,2026-11-04T06:12:58.000Z,STREAMING UPDATE,"Map(numRemovedFiles -> 0, numOutputRows -> 103, numOutputBytes -> 6054, numAddedFiles -> 1)"
1,2026-11-04T06:11:43.000Z,STREAMING UPDATE,"Map(numRemovedFiles -> 0, numOutputRows -> 102, numOutputBytes -> 6264, numAddedFiles -> 1)"
0,2026-11-04T06:11:34.000Z,CREATE TABLE,Map()
